In [ ]:
# Bibliotecas utilizadas
import pyodbc as odbc
import pandas as pd
import warnings
import os

# Pega usuário de rede
usuario = os.getenv("USERNAME")

# Retirando Warnings
warnings.filterwarnings('ignore')

# Conexão com o DW
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)
df_senhas = pd.read_excel('SENHAS.xlsx')
server = df_senhas.iloc[0,0]
database = df_senhas.iloc[0,1]
username = df_senhas.iloc[0,2]
password = df_senhas.iloc[0,3]
conn = odbc.connect(f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}')

# Avaliação por N formas
def tratando_documento(val):
    val = float(val)
    val = str(val)
    val = val.split('.')[0]
    return(val)

# Busca Rating
def faixas_score_pf():
    intervalos = [0, 337, 416, 476, 524, 567, 607, 651, 701, 1000]
    lista_rating = ['H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', ' AA']
    return intervalos, lista_rating

def faixas_score_pj():
    intervalos = [0, 187, 281, 359, 426, 495, 562, 631, 706, 1000]
    lista_rating = ['H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', ' AA']
    return intervalos, lista_rating

def define_rating(score, intervalos, lista_rating):
    for i in range(len(intervalos) - 1):
        if intervalos[i] < score <= intervalos[i + 1]:
            return lista_rating[i]
    return None  # Caso o score não se encaixe em nenhuma faixa

# Script para dados gerais
script = """
select
    a.ID_Cota,
    a.DT_Alocacao,
    a.Tipo_Pessoa,
    b.CD_InscricaoNacional as CPF_CNPJ
from [dbo].[FT0002_VendaAlocacoes] a
    left join [dbo].[DM0013_Pessoas] b on a.ID_Pessoa = b.ID_Pessoa
"""
df = pd.read_sql(script, conn)
df['CPF_CNPJ'] = df['CPF_CNPJ'].apply(tratando_documento)
df['DT_Alocacao'] = pd.to_datetime(df['DT_Alocacao'], format='%Y-%m-%d')

# Carregando Score de Crédito dos clientes
path = fr'C:\Users\{usuario}\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\MODELOS\MODELAGEM\SCORE_CREDITO_BOA_VISTA\BASES_HISTORICAS'
os.chdir(path)
df_score = pd.read_csv('score_credito_full.csv', delimiter=";")
df_score = df_score[df_score['Score_Credito'] > 2]
df_score['CPF_CNPJ'] = df_score['CPF_CNPJ'].apply(tratando_documento)
df_score = df_score[['Data', 'CPF_CNPJ', 'Score_Credito']]
df_score['Data'] = pd.to_datetime(df_score['Data'], format='%Y-%m-%d')

# União das tabelas
df = df.merge(df_score, how='left', on='CPF_CNPJ')

# Subtração de datas
df['Diferenca_Datas'] = abs(df['DT_Alocacao'] - df['Data']).dt.days

# Fazendo a média por ID_Cota
df = df[['ID_Cota', 'Tipo_Pessoa', 'Diferenca_Datas', 'Score_Credito']].groupby(by=['ID_Cota', 'Tipo_Pessoa', 'Diferenca_Datas']).mean().reset_index(level=['ID_Cota', 'Tipo_Pessoa', 'Diferenca_Datas'])

# Colocando a menor diferença de datas
df = df.sort_values(by=['Diferenca_Datas'], ascending=True)
df = df.drop_duplicates(subset=['ID_Cota'])

# Retirando casos acima de 30 dias
df = df[df['Diferenca_Datas'] <= 30]

teste2 = df.copy()

# Mantendo somente colunas de interesse
df = df[['ID_Cota', 'Tipo_Pessoa', 'Score_Credito']]

# Removendo casas decimais de Score_Credito (mantendo nulos)
df['Score_Credito'] = df['Score_Credito'].apply(lambda x: int(x) if pd.notnull(x) else x)

# Declarando Ratings e Scores
intervalos_pf = faixas_score_pf()[0]
lista_rating_pf = faixas_score_pf()[1]
intervalos_pj = faixas_score_pj()[0]
lista_rating_pj = faixas_score_pj()[1]

# Separando o df por produto
df_pf = df[df['Tipo_Pessoa'] == 'F']
df_pf['Rating'] = df_pf['Score_Credito'].apply(define_rating, intervalos=intervalos_pf, lista_rating=lista_rating_pf)
df_pj = df[df['Tipo_Pessoa'] == 'J']
df_pj['Rating'] = df_pj['Score_Credito'].apply(define_rating, intervalos=intervalos_pj, lista_rating=lista_rating_pj)

# Unindo os dois DataFrames
df = pd.concat([df_pf, df_pj])


KeyError: "['CD_InscricaoNacional'] not in index"

In [19]:
import pyodbc as odbc
import pandas as pd
import warnings
import os

# Retirando Warnings
warnings.filterwarnings('ignore')

# Pega usuário de rede
usuario = os.getenv("USERNAME")

# Caminho da planilha com senha
path_senha = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path_senha)

# Lê arquivo de senha
df_senhas = pd.read_excel('SENHAS.xlsx')

server = df_senhas.iloc[0,0]
database = df_senhas.iloc[0,1]
username = df_senhas.iloc[0,2]
password = df_senhas.iloc[0,3]

print(f"Tentando conectar ao servidor: {server} com usuário: {username}")

try:
    conn = odbc.connect(f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}', timeout=10)
    print("Conexão realizada com sucesso!")
except odbc.Error as e:
    print("Erro ao conectar ao banco de dados:")
    print(e)
    raise

# Funções para tratamento
def tratando_documento(val):
    val = float(val)
    val = str(val)
    val = val.split('.')[0]
    return val

def faixas_score_pf():
    intervalos = [0, 337, 416, 476, 524, 567, 607, 651, 701, 1000]
    lista_rating = ['H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', ' AA']
    return intervalos, lista_rating

def faixas_score_pj():
    intervalos = [0, 187, 281, 359, 426, 495, 562, 631, 706, 1000]
    lista_rating = ['H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', ' AA']
    return intervalos, lista_rating

def define_rating(score, intervalos, lista_rating):
    for i in range(len(intervalos) - 1):
        if intervalos[i] < score <= intervalos[i + 1]:
            return lista_rating[i]
    return None

# Script SQL para dados gerais
script = """
select
    a.ID_Cota,
    a.DT_Alocacao,
    a.Tipo_Pessoa,
    b.CD_InscricaoNacional as CPF_CNPJ
from [dbo].[FT0002_VendaAlocacoes] a
    left join [dbo].[DM0013_Pessoas] b on a.ID_Pessoa = b.ID_Pessoa
"""
df = pd.read_sql(script, conn)

# Mantendo CD_InscricaoNacional antes da limpeza
df['CD_InscricaoNacional'] = df['CPF_CNPJ']

df['CPF_CNPJ'] = df['CPF_CNPJ'].apply(tratando_documento)
df['DT_Alocacao'] = pd.to_datetime(df['DT_Alocacao'], format='%Y-%m-%d')

# Carregando Score de Crédito dos clientes
path_score = fr'C:\Users\{usuario}\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\MODELOS\MODELAGEM\SCORE_CREDITO_BOA_VISTA\BASES_HISTORICAS'
os.chdir(path_score)
df_score = pd.read_csv('score_credito_full.csv', delimiter=";")
df_score = df_score[df_score['Score_Credito'] > 2]
df_score['CPF_CNPJ'] = df_score['CPF_CNPJ'].apply(tratando_documento)
df_score = df_score[['Data', 'CPF_CNPJ', 'Score_Credito']]
df_score['Data'] = pd.to_datetime(df_score['Data'], format='%Y-%m-%d')

# União das tabelas
df = df.merge(df_score, how='left', on='CPF_CNPJ')

# Subtração de datas
df['Diferenca_Datas'] = abs(df['DT_Alocacao'] - df['Data']).dt.days

# Agrupar e calcular média somente nas colunas numéricas
df_agg = df[['ID_Cota', 'Tipo_Pessoa', 'Diferenca_Datas', 'Score_Credito']].groupby(
    ['ID_Cota', 'Tipo_Pessoa', 'Diferenca_Datas']
).mean().reset_index()

# Recuperar CD_InscricaoNacional único por ID_Cota (texto)
cd_inscricao = df[['ID_Cota', 'CD_InscricaoNacional']].drop_duplicates(subset=['ID_Cota'])

# Juntar CD_InscricaoNacional ao dataframe agregado
df_agg = df_agg.merge(cd_inscricao, on='ID_Cota', how='left')

# Ordenar e remover duplicados mantendo menor diferença
df_agg = df_agg.sort_values(by=['Diferenca_Datas'], ascending=True)
df_agg = df_agg.drop_duplicates(subset=['ID_Cota'])

# Filtrar diferenças até 30 dias
df_agg = df_agg[df_agg['Diferenca_Datas'] <= 30]

# Mantendo somente colunas de interesse
df_final = df_agg[['ID_Cota', 'Tipo_Pessoa', 'Score_Credito', 'CD_InscricaoNacional']]

# Removendo casas decimais de Score_Credito (mantendo nulos)
df_final['Score_Credito'] = df_final['Score_Credito'].apply(lambda x: int(x) if pd.notnull(x) else x)

# Declarando Ratings e Scores
intervalos_pf, lista_rating_pf = faixas_score_pf()
intervalos_pj, lista_rating_pj = faixas_score_pj()

# Separando o df por produto
df_pf = df_final[df_final['Tipo_Pessoa'] == 'F'].copy()
df_pf['Rating'] = df_pf['Score_Credito'].apply(define_rating, args=(intervalos_pf, lista_rating_pf))

df_pj = df_final[df_final['Tipo_Pessoa'] == 'J'].copy()
df_pj['Rating'] = df_pj['Score_Credito'].apply(define_rating, args=(intervalos_pj, lista_rating_pj))

# Unindo os dois DataFrames
df_final = pd.concat([df_pf, df_pj], ignore_index=True)


Tentando conectar ao servidor: Server227,1440 com usuário: CXE10160
Conexão realizada com sucesso!


In [17]:
import pandas as pd
import os

# Caminho base onde está o Excel e onde vai salvar o Parquet
caminho_base = r"C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\INDICADORES\NOVA ESTRUTURA\INADIMPLÊNCIA\INADIMPLÊNCIA GERAL\Dados\Dados - Ramo de Atividade\Principal"

# Leitura do Excel com os dados já prontos
df_final = pd.read_excel(os.path.join(caminho_base, "df_final_inadimplencia.xlsx"))

# Leitura do arquivo parquet (ajuste o caminho/nome do arquivo parquet)
arquivo_parquet = os.path.join(caminho_base, "arquivo_unificado.parquet")  # <== Ajuste aqui o nome do arquivo parquet original

df_parquet = pd.read_parquet(arquivo_parquet)

# Criar coluna concatenada CNPJ completo como texto, removendo zeros à esquerda se necessário (normalmente CNPJ é 14 dígitos)
# Concatenando as colunas como strings com zeros à esquerda para garantir tamanho fixo
df_parquet['CNPJ_completo'] = (
    df_parquet['CNPJ_BASICO'].astype(str).str.zfill(8) +
    df_parquet['CNPJ_ORDEM'].astype(str).str.zfill(4) +
    df_parquet['CNPJ_DV'].astype(str).str.zfill(2)
)

# O CNPJ na base 'df_final' pode estar no mesmo formato de 'CD_InscricaoNacional', então vamos garantir que os dois estejam no mesmo padrão:
df_final['CD_InscricaoNacional'] = df_final['CD_InscricaoNacional'].astype(str).str.zfill(14)

# Merge para trazer o Rating pelo CNPJ completo
df_merged = df_parquet.merge(df_final[['CD_InscricaoNacional', 'Rating']],
                            left_on='CNPJ_completo', right_on='CD_InscricaoNacional',
                            how='left')

# Salvar o dataframe resultante como parquet
caminho_salvar = os.path.join(caminho_base, "COMPLETO.parquet")
df_merged.to_parquet(caminho_salvar, index=False)

print(f"Arquivo COMPLETO.parquet salvo em: {caminho_salvar}")


ArrowMemoryError: realloc of size 2097152 failed

In [7]:
df_final.head()

,ID_Cota,Tipo_Pessoa,Score_Credito,CD_InscricaoNacional,Rating
2547200,197273,F,437,00543856100,F
2542354,196934,F,525,18043232091,D
2547225,197275,F,358,20236859900,G
2542328,196932,F,63,71313037206,H
2542341,196933,F,630,89028171991,B
